BGE-m3-multivector-initialization

In [1]:
from FlagEmbedding import BGEM3FlagModel
from pymilvus import (
    MilvusClient,connections,FieldSchema,CollectionSchema,DataType,Collection,RRFRanker,utility,AnnSearchRequest
)
from datasets import Dataset
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List,Dict,Tuple
from datasets import load_dataset

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"]=(12,6)

/Users/nilasark/advanced/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:


documents = [
    "Global warming is primarily caused by increased greenhouse gas emissions from human activities, particularly the burning of fossil fuels like coal, oil, and natural gas.",
    "Over the coming 25 or 30 years, scientists say, the climate is likely to gradually warm. However, researchers also say that this phenomenon can be stopped if human emissions are reduced to zero.",
    "The jet stream forms a boundary between the cold north and the warmer south, but the lower temperature difference means the winds are now weaker, leading to more extreme weather patterns.",
    "Coral reefs become stressed due to ocean acidification and warming, expelling their symbiotic algae which leaves the coral a bleached white color. This process threatens entire marine ecosystems.",
    "The rapid changes in the climate may have profound consequences for humans and other species. Severe drought caused food shortages for millions of people in Ethiopia, with a lack of rainfall resulting in intense and widespread forest fires.",
    "Rising sea levels threaten coastal cities worldwide, with predictions suggesting that many major urban centers could face significant flooding by 2100 if current trends continue.",
    "Melting Arctic ice reduces the Earth's albedo effect, causing the planet to absorb more solar radiation and accelerating the warming process in a dangerous feedback loop.",
    "Climate change is disrupting agricultural patterns, forcing farmers to adapt their crops and techniques to new temperature and precipitation regimes that differ from historical norms."
]

print(f"Loaded {len(documents)} documents")
print(f"Sample Documents")
for idx,doc in enumerate(documents,1):
    print(f"{idx}: {doc[:200]}")



Loaded 8 documents
Sample Documents
1: Global warming is primarily caused by increased greenhouse gas emissions from human activities, particularly the burning of fossil fuels like coal, oil, and natural gas.
2: Over the coming 25 or 30 years, scientists say, the climate is likely to gradually warm. However, researchers also say that this phenomenon can be stopped if human emissions are reduced to zero.
3: The jet stream forms a boundary between the cold north and the warmer south, but the lower temperature difference means the winds are now weaker, leading to more extreme weather patterns.
4: Coral reefs become stressed due to ocean acidification and warming, expelling their symbiotic algae which leaves the coral a bleached white color. This process threatens entire marine ecosystems.
5: The rapid changes in the climate may have profound consequences for humans and other species. Severe drought caused food shortages for millions of people in Ethiopia, with a lack of rainfall resulting

In [4]:
print("Loading BGE-M3 model ...")

model=BGEM3FlagModel(
    model_name_or_path="BAAI/bge-m3",
    devices="cpu"
)
print("BGE-M3 model successfully loaded")

Loading BGE-M3 model ...


Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 371177.35it/s]


BGE-M3 model successfully loaded


BGE-m3 embeddings generated

In [7]:
print(f"Sparse,Dense and ColBERT embeddings will be genrated")

doc_embeddings=model.encode(
    documents,
    batch_size=1000,
    max_length=512,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False
)
dense_embeddings=doc_embeddings['dense_vecs']
sparse_embeddings=doc_embeddings['lexical_weights']

print(f"Dense Vector Shape : {dense_embeddings.shape}")
print(f"Sparse Embedding Size: {len(sparse_embeddings)} length")



Sparse,Dense and ColBERT embeddings will be genrated
Dense Vector Shape : (8, 1024)
Sparse Embedding Size: 8 length


In [16]:
connections.connect(
    alias="default",
    uri="http://localhost:19530"
)

print(f"Connected To Milvus")
collection_name="bge_m3_hybrid"

if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)
    print(f"Dropping Collection")

fields=[
    FieldSchema(name='pk',dtype=DataType.VARCHAR,is_primary=True,auto_id=True,max_length=100),
    FieldSchema(name="text",dtype=DataType.VARCHAR,max_length=2000),
    FieldSchema(name="dense_vector",dtype=DataType.FLOAT_VECTOR,dim=1024),
    FieldSchema(name='sparse_vector',dtype=DataType.SPARSE_FLOAT_VECTOR)
]

schema=CollectionSchema(
    fields=fields,
    description="BGE-M3 multi-vector hybrid retrieval demo"
)

collection=Collection(
    name=collection_name,
    schema=schema
)

print(f"Collection {collection_name} created")

Connected To Milvus
Collection bge_m3_hybrid created


/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_19987/1890051003.py:1: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(
/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_19987/1890051003.py:9: PyMilvusDeprecationWarning: `utility.has_collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  if utility.has_collection(collection_name):
/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_19987/1890051003.py:25: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection=Collection(
